# 07a — Trip Duration EDA (for agent model tuning)

Quick exploration of `trip_duration_min` distribution to decide:
1. What cap/filter to apply before training the duration model
2. Whether log-transform is appropriate

**Run all cells and share the output.**

In [ ]:
from pyspark.sql import functions as F

from src.constants import SILVER_TABLE

silver_df = spark.read.table(SILVER_TABLE)
print(f"Silver rows: {silver_df.count():,}")

In [ ]:
# ── Basic stats ───────────────────────────────────────────────────────────────
dur = silver_df.select("trip_duration_min")

dur.describe().show()

# Percentiles: 1st, 5th, 10th, 25th, 50th, 75th, 90th, 95th, 99th, 99.5th, 99.9th
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
pct_values = dur.approxQuantile("trip_duration_min", percentiles, 0.001)

print("\nPercentile distribution of trip_duration_min:")
for p, v in zip(percentiles, pct_values):
    print(f"  P{p * 100:5.1f}:  {v:8.1f} min")

In [ ]:
# ── How many rows at various cap thresholds ──────────────────────────────────
total = silver_df.count()

thresholds = [30, 45, 60, 90, 120, 180, 360, 1440]
print(f"{'Cap (min)':>10}  {'Rows kept':>12}  {'% kept':>8}  {'Rows dropped':>12}")
print("-" * 50)
for t in thresholds:
    kept = silver_df.filter(
        (F.col("trip_duration_min") > 0) & (F.col("trip_duration_min") <= t)
    ).count()
    print(f"{t:>10}  {kept:>12,}  {kept / total * 100:>7.2f}%  {total - kept:>12,}")

In [ ]:
# ── Duration by common segments ──────────────────────────────────────────────
# Breakdown: how many trips fall into each duration bucket?
silver_df.filter(F.col("trip_duration_min") > 0).withColumn(
    "duration_bucket",
    F.when(F.col("trip_duration_min") <= 5, "0-5 min")
    .when(F.col("trip_duration_min") <= 15, "5-15 min")
    .when(F.col("trip_duration_min") <= 30, "15-30 min")
    .when(F.col("trip_duration_min") <= 60, "30-60 min")
    .when(F.col("trip_duration_min") <= 120, "60-120 min")
    .otherwise("120+ min"),
).groupBy("duration_bucket").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"),
    F.round(F.avg("total_amount"), 2).alias("avg_fare"),
).orderBy("duration_bucket").show(truncate=False)

In [ ]:
# ── Negative and zero durations ──────────────────────────────────────────────
neg_zero = silver_df.filter(F.col("trip_duration_min") <= 0).count()
print(f"Rows with trip_duration_min <= 0: {neg_zero:,} ({neg_zero / total * 100:.3f}%)")

# ── NULLs ─────────────────────────────────────────────────────────────────────
nulls = silver_df.filter(F.col("trip_duration_min").isNull()).count()
print(f"Rows with NULL trip_duration_min:  {nulls:,}")